# In this file, we try to train a unet model. However, this currently can only be done with python script, so this file we load the self-trained models. Then, we test to see how well it performs.

In [ ]:
from config_io import *
from train.train import run
from DNAnet.data.kit_compatibility.lane_standards import InternalSizeStandard

/Users/amarmesic/miniconda3/envs/dnanet/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from DNAnet.data.data_models.dna_models import Panel
from DNAnet.data.parsing.file_categorization_strategy import ProvedItFileCategorizer
from DNAnet.data.parsing.file_parsing import find_files_by_suffix
from DNAnet.data.validation.sample_validation_strategy import NFIValidationStrategy

gf_panel_path = "resources/data/ProvedIt/SGPanel_Globalfiler_Panel.xml"
hid_files_path = "/Users/amarmesic/Documents/tudelft/thesis/datasets/USE THIS - PROVEDIt_2-5-Person Profiles_3500 5sec_GF29cycles"
reference_genotype_path = "/Users/amarmesic/Documents/tudelft/thesis/DNANet/resources/data/ProvedIt/References"

panel = Panel(gf_panel_path)
hid_files = find_files_by_suffix(hid_files_path, ".hid")
file_categorizer = ProvedItFileCategorizer()
sample_validator = NFIValidationStrategy()

In [4]:
dataset = CustomHIDDataset(
    files=hid_files,
    panel=panel,
    size_standard=InternalSizeStandard.GENESCAN_600_LIZ.value,
    file_categorization_strategy=file_categorizer,
    sample_validation_strategy=sample_validator,
)

2025-06-18 09:37:53 WARNING  Size standard peaks validation failed. Trying validation only scan points after 4000
2025-06-18 09:37:53 WARNING  Size standard peaks validation failed. Trying validation only scan points after 4000
2025-06-18 09:37:53 WARNING  Size standard peaks validation failed. Trying validation only scan points after 4000
2025-06-18 09:37:53 WARNING  Size standard peaks validation failed. Trying validation only scan points after 4000
2025-06-18 09:37:53 WARNING  Size standard peaks validation failed. Trying validation only scan points after 4000
2025-06-18 09:37:53 WARNING  Size standard peaks validation failed. Trying validation only scan points after 4000
2025-06-18 09:37:53 WARNING  Size standard peaks validation failed. Trying validation only scan points after 4000
2025-06-18 09:37:53 WARNING  Size standard peaks validation failed. Trying validation only scan points after 4000
2025-06-18 09:37:53 WARNING  Size standard peaks validation failed. Trying validation on

# Call run

In [5]:
run(
    data_config=dataset,                # -d
    model_config="unet_amar_1",                # -m
    training_config="training_amar",    # -t
    split=0.9,                          # -s # implement custom split
    validation_config=0.1,              # -v
    output_dir="output/example_25-06-17_1st",  # -o
    seed=42                            # -rs
)

2025-06-18 09:37:58 INFO     Logs will be written to output/example_25-06-17_1st/log_training.txt
2025-06-18 09:37:58 INFO     Loading model...
2025-06-18 09:37:58 INFO     Will start training from scratch
2025-06-18 09:37:58 INFO     Loading dataset...
2025-06-18 09:37:58 INFO     Using provided dataset directly
2025-06-18 09:37:58 INFO     Splitting dataset, using 90.0% for training
2025-06-18 09:37:58 INFO     Validation split found, using 10.0% to create a validation set...
2025-06-18 09:37:58 INFO     Starting training...
2025-06-18 09:37:58 INFO     Setting up exponential scheduler, starting with learning rate 0.0001
2025-06-18 09:37:58 INFO     Tensorboard logs are written to output/example_25-06-17_1st/tensorboard
2025-06-18 09:37:58 INFO     Run `tensorboard --logdir=output/example_25-06-17_1st/tensorboard`
Epoch 1/4: 100%|██████████| 7/7 [00:01<00:00,  4.95it/s, training_loss=0.956]
2025-06-18 09:37:59 INFO     Epoch 1/4 - Training binaryaccuracy: 0.935722
2025-06-18 09:37:59

TypeError: stat: path should be string, bytes, os.PathLike or integer, not CustomHIDDataset

In [ ]:
# Load your model config (same one used for training)
amar_model = load_model("config/models/unet.yaml")

# Load the trained weights
amar_model.load("output/example_run_train/")

amar_model_predictions = amar_model.predict_batch(dataset)

2025-06-17 18:16:51 INFO     Calling alleles from predicted segmentation...


In [ ]:
print('allele u-net f1: ', allele_f1_score(dataset, amar_model_predictions))
print('allele u-net precision: ', allele_precision(dataset, amar_model_predictions))
print('allele u-net recall: ', allele_recall(dataset, amar_model_predictions))

allele u-net f1:  0.6067861392564815
allele u-net precision:  0.4488663643638304
allele u-net recall:  0.9361370716510904


In [ ]:
# Load your model config (same one used for training)
best_model = load_model("config/models/unet.yaml")

# Load the trained weights
best_model.load("resources/model/current_best_unet/")

best_model_predictions = best_model.predict_batch(dataset)

2025-06-18 09:19:56 INFO     Calling alleles from predicted segmentation...
2025-06-18 09:19:56 WARNING  No predictions present in dye row 3


In [ ]:
print('allele u-net f1: ', allele_f1_score(dataset, best_model_predictions))
print('allele u-net precision: ', allele_precision(dataset, best_model_predictions))
print('allele u-net recall: ', allele_recall(dataset, best_model_predictions))

allele u-net f1:  0.8044332375535097
allele u-net precision:  0.8504649721016739
allele u-net recall:  0.7631286159323543
